# Sprint E12 walkthrough: attribution, and the skill question

Purpose: a mechanistic explanation of every estimator the sprint built,
reproduced by hand, so each number on the page can be derived at a
whiteboard and the memo's numbers have an audit trail.

The three research questions, from the roadmap:

1. Where did the P&L come from, factor by factor, and does the split add back
   up to the total to machine precision?
2. Do the two ways of measuring the same thing, holdings and returns,
   agree?
3. Is the book any good, or is it luck?

The intuition, in one paragraph. Attribution is the risk decomposition run
**backwards** on realized returns: every basis point the book earned is a
factor basis point, an idiosyncratic basis point, or a cost, and the
reconciliation is what makes that a check rather than a restatement. The
holdings view knows what the book held; the regression view only sees what it
earned; when they disagree, the exposures moved between rebalances. And the
skill question has an arithmetic answer that has nothing to do with how the
book feels: a t-statistic, and the number of days an edge of a given size
needs before the test can see it at all.

In [1]:
import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "efb").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
ATTRIBUTION = DATA / "attribution"

## 1. The artifact, and the criteria as they were registered

The attribution is stored per session by `scripts/build_attribution.py`, from
a book's own stored weights. Here it is the seed book's: rho 0.02, seed 0 out
of `data/portfolios/mv_constrained.parquet`. The live book swaps that input
and nothing else.

In [2]:
daily = pd.read_parquet(ATTRIBUTION / 'daily.parquet')
monthly = pd.read_parquet(ATTRIBUTION / 'monthly.parquet')
timeseries = pd.read_parquet(ATTRIBUTION / 'timeseries.parquet')
daily['trade_date'] = pd.to_datetime(daily['trade_date'])
results = json.loads((ROOT / 'sprints' / 'E12' / 'RESULTS.json').read_text())
print('INPUT  data/attribution/daily.parquet:', daily.shape,
      '| monthly:', monthly.shape, '| timeseries:', timeseries.shape)
print('OUTPUT sprints/E12/RESULTS.json:')
for name, block in results['criteria'].items():
    print(f'  {name}: {block["verdict"]}')
print('  F12.3 expected verdict:', results['criteria']['F12.3']['expected_verdict'])
print('  over', results['artifact']['n_sessions'], 'sessions:',
      results['artifact']['first_session'], 'to', results['artifact']['last_session'])

INPUT  data/attribution/daily.parquet: (3645, 29) | monthly: (174, 15) | timeseries: (20, 5)
OUTPUT sprints/E12/RESULTS.json:
  F12.1: partial
  F12.2: pending
  F12.3: pending
  F12.3 expected verdict: luck
  over 3645 sessions: 2012-02-01 to 2026-07-31


## 2. The identity, by hand on two names

The formulas the sprint implements, with every symbol labelled:

- factor P&L $_t = (X_{t}\' w_{t-1})\' f_t$, **INPUT** `descriptors.parquet`
  and `factor_returns.parquet`, **OUTPUT** `daily.pnl_factor_json`.
- idio P&L $_t = w_{t-1}\' u_t$, **INPUT** `specific_returns.parquet`,
  **OUTPUT** `daily.pnl_idio`.
- total $_t$ = factor + idio + cost, **OUTPUT** `daily.identity_residual`.

Two names, one factor, numbers a reader can check on paper: weights (0.6,
-0.4), returns (0.02, -0.01), design X = (1.0, -0.5), factor return 0.01, cost
-0.001.

In [3]:
from efb import attribution

date = pd.Timestamp('2026-09-21')
holdings = pd.DataFrame({'date': [date] * 2, 'ticker': ['AAA', 'BBB'],
                         'weight': [0.6, -0.4]})


class OneFactorPanel:
    """A panel small enough to do on paper: one factor, two names."""

    def __init__(self):
        self.returns = pd.DataFrame({'AAA': [0.02], 'BBB': [-0.01]},
                                    index=[date]).stack(future_stack=True)
        self.factor_returns = pd.DataFrame({'market': [0.01]}, index=[date])
        self._design = np.array([[1.0], [-0.5]])

    def design(self, stamp, names):
        return self._design

    def raw_design(self, stamp, names):
        return self._design

    def reported_design(self, stamp, names):
        return self._design

    def factor_names(self, stamp):
        return ['market']

    def betas_at(self, stamp, names):
        return np.array([1.2, 0.4])

    def fit_or_stored(self, stamp, names, returns, raw):
        fitted = self._design @ self.factor_returns.loc[stamp].to_numpy(float)
        return self.factor_returns.loc[stamp], returns - fitted, 'fixture'


row = attribution.attribute_book(holdings, OneFactorPanel(),
                                costs={date: {'cost_usd': -0.001}}).iloc[0]
exposure = 0.6 * 1.0 + (-0.4) * (-0.5)
factor_pnl = exposure * 0.01
gross_pnl = 0.6 * 0.02 + (-0.4) * (-0.01)
print(f'  exposure X\'w      = {exposure:+.4f}')
print(f'  factor P&L         = {factor_pnl:+.6f}  (library {row["pnl_factor"]:+.6f})')
print(f'  gross P&L          = {gross_pnl:+.6f}')
print(f'  idio P&L           = {gross_pnl - factor_pnl:+.6f}  '
      f'(library {row["pnl_idio"]:+.6f})')
print(f'  cost               = {row["pnl_cost"]:+.6f}')
print(f'  total              = {row["pnl_total"]:+.6f}')
print(f'  identity residual  = {row["identity_residual"]:.1e}  '
      f'(tolerance {attribution.IDENTITY_ATOL:.0e})')
assert row['pnl_factor'] == row['pnl_factor']
assert abs(row['pnl_total'] - (factor_pnl + (gross_pnl - factor_pnl) + (-0.001))) < 1e-15
assert row['identity_residual'] == 0.0

  exposure X'w      = +0.8000
  factor P&L         = +0.008000  (library +0.008000)
  gross P&L          = +0.016000
  idio P&L           = +0.008000  (library +0.008000)
  cost               = -0.001000
  total              = +0.015000
  identity residual  = 0.0e+00  (tolerance 1e-10)


## 3. The identity on the whole artifact

The same check, on every session the artifact carries, where the three
components come from three different stored artifacts rather than from a
fixture. The residual is the check; it is not asserted to be small, it is
measured.

In [4]:
residual = daily['identity_residual'].abs()
print('INPUT  data/attribution/daily.parquet')
print(f'  sessions            {len(daily):,}')
print(f'  median |residual|   {residual.median():.3e}')
print(f'  max    |residual|   {residual.max():.3e}')
print(f'  sessions inside 1e-10: {int((residual < attribution.IDENTITY_ATOL).sum()):,}')
split = daily['pnl_factor_json'].apply(lambda m: float(sum(m.values())))
assert np.allclose(split, daily['pnl_factor'], atol=1e-12)
print('  the per-factor split sums to the factor total on every session: yes')

INPUT  data/attribution/daily.parquet
  sessions            3,645
  median |residual|   1.771e-04
  max    |residual|   1.233e-02
  sessions inside 1e-10: 174
  the per-factor split sums to the factor total on every session: yes


## 4. The cumulative split, and the five largest months

Cumulative answers whether it worked; by period answers when. The book is
gross 1.0 of NAV, so a basis point of the book is a basis point of NAV.

In [5]:
totals = {name: float(daily[column].sum()) for name, column in
          (('total', 'pnl_total'), ('factor', 'pnl_factor'),
           ('idio', 'pnl_idio'), ('cost', 'pnl_cost'),
           ('hedge timing', 'pnl_timing'))}
for name, value in totals.items():
    print(f'  {name:14s} {value * 1e4:+10.1f} bp')
by_factor = {}
for mapping in daily['pnl_factor_json']:
    for name, value in mapping.items():
        by_factor[name] = by_factor.get(name, 0.0) + float(value)
ranked = sorted(by_factor.items(), key=lambda pair: -abs(pair[1]))
print('  by factor (largest first):')
for name, value in ranked[:6]:
    print(f'    {name:12s} {value * 1e4:+10.1f} bp')
monthly = monthly.copy()
monthly['magnitude'] = monthly['pnl_total'].abs()
print('  the five largest months:')
for record in monthly.sort_values('magnitude', ascending=False).head(5).to_dict('records'):
    print(f'    {record["month"]}  n={int(record["n_sessions"]):2d}  '
          f'total {record["pnl_total"] * 1e4:+9.1f} bp  '
          f'factor {record["pnl_factor"] * 1e4:+8.1f}  '
          f'idio {record["pnl_idio"] * 1e4:+9.1f}')

  total             +7039.6 bp
  factor              -33.0 bp
  idio              +7466.7 bp
  cost                 +0.0 bp
  hedge timing       +141.4 bp
  by factor (largest first):
    liquidity        -170.9 bp
    reversal          +81.4 bp
    momentum          +55.5 bp
    size              +16.1 bp
    resid_vol          -7.6 bp
    beta               -7.5 bp
  the five largest months:
    2025-02  n=19  total    -462.0 bp  factor     -7.7  idio    -465.1
    2020-03  n=22  total    +431.7 bp  factor    -29.1  idio    +560.1
    2017-05  n=22  total    -424.4 bp  factor     -9.9  idio    -385.6
    2018-05  n=22  total    +423.5 bp  factor    +12.3  idio    +415.6
    2026-05  n=20  total    +405.8 bp  factor     +9.3  idio    +362.2


## 5. The hedge timing, and which design vintage the model uses

Two facts, and they are separate. **The design dated a session is built from
data through the previous close**, so it is computable at that close and the
stored model has no look-ahead. The hedge is nonetheless one session
**behind**, because the exposure the book will be measured on is the design
dated the next session, which is also available at the same close.

The first fact, from the code and then from the artifact: size is
`log(market_cap.shift(1))`, and beta, residual volatility and liquidity each
carry `.shift(1)` on their window; reversal shifts the returns by one and only
momentum reaches further back.

In [6]:
from efb.models import fundamental as fx

prices = pd.read_parquet(DATA / 'raw' / 'prices.parquet')
close = prices['close'].unstack('ticker')
shares = pd.read_parquet(DATA / 'raw' / 'shares_history.parquet')
shares = shares.pivot_table(index='date', columns='ticker', values='shares')
shares = shares.reindex(close.index).ffill()
mcap = fx.market_cap(close, shares)
descriptors = pd.read_parquet(DATA / 'models' / 'XS-v1' / 'descriptors.parquet')
size = descriptors.loc[descriptors['descriptor'] == 'size']
stamp = pd.Timestamp('2026-09-21')
previous = pd.DatetimeIndex(sorted(set(mcap.index)))
previous = previous[previous < stamp][-1]
row = size.loc[pd.to_datetime(size['date']) == stamp].set_index('ticker')['value_raw']
for ticker in ('LITE', 'MRNA', 'AAPL'):
    same = math.log(float(mcap.loc[stamp, ticker]))
    lagged = math.log(float(mcap.loc[previous, ticker]))
    print(f'  {ticker:5s} size descriptor {row[ticker]:+.6f} | at t gap '
          f'{abs(row[ticker] - same):.3e} | at t-1 gap '
          f'{abs(row[ticker] - lagged):.3e}')
print('  -> the row dated t is the previous close, so no look-ahead.')

  LITE  size descriptor +25.148144 | at t gap 2.501e-02 | at t-1 gap 0.000e+00
  MRNA  size descriptor +24.842275 | at t gap 1.157e-01 | at t-1 gap 0.000e+00
  AAPL  size descriptor +29.221387 | at t gap 8.443e-03 | at t-1 gap 0.000e+00
  -> the row dated t is the previous close, so no look-ahead.


In [7]:
# The second fact: the vintages differ on the sessions after a rebalance, and
# the drift between them is what the hedge left un-neutralized.
moved = daily.loc[daily['pnl_timing'].abs() > 0]
gap = pd.DataFrame(list(moved['exposure_json'])) - pd.DataFrame(
    list(moved['book_exposure_json']))
print(f'  sessions where the two vintages differ: {len(moved):,} of {len(daily):,}')
print(f'  mean |per-factor gap| on those       : {gap.abs().mean().mean():.5f}')
print(f'  largest single gap                   : {gap.abs().max().max():.4f} '
      f'of {gap.abs().max().idxmax()}')
print(f'  timing P&L                           : '
      f'{float(moved["pnl_timing"].sum()) * 1e4:+.1f} bp')
print(f'  total P&L on those sessions          : '
      f'{float(moved["pnl_total"].sum()) * 1e4:+.1f} bp')
print(f'  the timing line as a share of it     : '
      f'{abs(float(moved["pnl_timing"].sum())) / abs(float(moved["pnl_total"].sum())) * 100:.1f}%')

  sessions where the two vintages differ: 174 of 3,645
  mean |per-factor gap| on those       : 0.01967
  largest single gap                   : 0.6258 of reversal
  timing P&L                           : +141.4 bp
  total P&L on those sessions          : +520.7 bp
  the timing line as a share of it     : 27.2%


## 6. The two estimators, side by side

The holdings view is the exposure times the factor return, per session. The
returns view is the book's own P&L regressed on the factor returns, with
standard errors from the regression's own covariance. F12.2 asks whether each
beta sits within one standard error of the average holdings exposure.

In [8]:
print('  term            beta       se   holdings   within 1 SE')
for record in timeseries.to_dict('records'):
    if pd.isna(record.get('holdings_exposure')):
        continue
    print(f'  {record["term"]:12s} {record["beta"]:+8.4f} {record["se"]:8.4f} '
          f'{record["holdings_exposure"]:+9.4f}   '
          f'{"yes" if record["within_one_se"] else "no"}')
print(f'  {int(timeseries["within_one_se"].sum())} of {len(timeseries)} terms agree  '
      f'(F12.2 is {results["criteria"]["F12.2"]["verdict"]})')
print('  the book rebalances monthly, so the regression absorbs a month of drift')
print('  in its intercept; the live book rebalances daily and is the object the')
print('  criterion is about.')

  term            beta       se   holdings   within 1 SE
  beta          +0.0092   0.0176   +0.0002   yes
  liquidity     +0.0028   0.0351   +0.0002   yes
  market        +0.0215   0.0077   -0.0000   no
  momentum      +0.0250   0.0208   -0.0000   no
  resid_vol     -0.0888   0.0305   +0.0004   no
  reversal      +0.0230   0.0206   +0.0021   no
  sector_10     -0.0158   0.0067   +0.0000   no
  sector_15     -0.0292   0.0112   -0.0000   no
  sector_20     +0.0140   0.0170   +0.0000   yes
  sector_25     -0.0067   0.0129   +0.0000   yes
  sector_30     -0.0225   0.0144   +0.0000   no
  sector_35     -0.0155   0.0159   +0.0000   yes
  sector_40     -0.0312   0.0187   +0.0000   no
  sector_45     -0.0579   0.0220   +0.0000   no
  sector_50     -0.0209   0.0126   +0.0000   no
  sector_55     -0.0015   0.0090   -0.0000   yes
  sector_60     -0.0040   0.0096   -0.0000   yes
  size          +0.0739   0.0337   +0.0002   no
  7 of 20 terms agree  (F12.2 is pending)
  the book rebalances monthly,

## 7. Skill against luck

The rule: no skill is claimed unless the t-statistic exceeds two, and the
Sharpe carries its standard error. Both standard errors come from `efb.perf`,
the E1 library, so this is the same estimator the E1 report used.

In [9]:
from efb import perf

skill = attribution.skill_test(daily)
total_series = daily['pnl_total']
print(f'  days                          {skill["n_days"]:,}')
print(f'  mean idio P&L                 {skill["idio_mean"] * 1e4:+.4f} bp/day')
print(f'  its standard error            {skill["idio_se"] * 1e4:.4f} bp')
print(f'  t-statistic                   {skill["t_stat"]:.2f}')
print(f'  annualized information ratio  {skill["ir_annual"]:.3f}')
print(f'  annualized Sharpe             {skill["sharpe_annual"]:.3f}')
print(f'    SE, i.i.d.                  {skill["sharpe_se_iid"]:.3f}')
print(f'    SE, Lo 2002                 {skill["sharpe_se_lo2002"]:.3f}')
assert skill['sharpe_annual'] == perf.annualized_sharpe(total_series)
assert skill['sharpe_se_lo2002'] == perf.sharpe_se_lo2002(total_series) * math.sqrt(252)
print()
print('  days needed for a t of', skill['target_t'], ':')
for ir, days in skill['days_to_detect'].items():
    print(f'    annualized IR {ir:4.2f} -> {days:>6,} days ({days / 252:5.1f} years)')
print()
print('  verdict:', skill['verdict'])
print('  E10 chose the (rho, phi) whose Sharpe is closest to 1.0 on this same')
print('  sample, so this is the selection working, not a discovery. The live book')
print('  trades a different, null signal; its expected verdict is luck.')

  days                          3,645
  mean idio P&L                 +2.0485 bp/day
  its standard error            0.5971 bp
  t-statistic                   3.43
  annualized information ratio  0.902
  annualized Sharpe             0.849
    SE, i.i.d.                  0.263
    SE, Lo 2002                 0.258

  days needed for a t of 2.0 :
    annualized IR 0.25 -> 16,128 days ( 64.0 years)
    annualized IR 0.50 ->  4,032 days ( 16.0 years)
    annualized IR 1.00 ->  1,008 days (  4.0 years)
    annualized IR 2.00 ->    252 days (  1.0 years)

  verdict: t = 3.43 over 3645 days exceeds 2.0, so the number clears the bar F12.3 sets
  E10 chose the (rho, phi) whose Sharpe is closest to 1.0 on this same
  sample, so this is the selection working, not a discovery. The live book
  trades a different, null signal; its expected verdict is luck.


## 8. What the page reads, panel by panel

`docs/snapshot.schema.json` is the contract between the writer and the page.
Every panel below is a key in it, and every key is fed by a column of the
stored artifact rather than by a recomputation.

In [10]:
from live import snapshot

block = snapshot.attribution_block(daily)
schema = json.loads((ROOT / 'docs' / 'snapshot.schema.json').read_text())
declared = schema['properties']['attribution']
print('  the schema declares attribution as required:', 'attribution' in schema['required'])
print('  the builder emits exactly its keys:', set(block) == set(declared['properties']))
print('  cumulative sums are the artifact\'s own sums:',
      math.isclose(block['cumulative']['pnl_total'], float(daily['pnl_total'].sum())))
print()
print('  panel                          source column')
for panel, column in (('cumulative total/factor/idio/cost', 'pnl_total, pnl_factor, pnl_idio, pnl_cost'),
                      ('worst day\'s identity residual', 'identity_residual'),
                      ('the hedge\'s factor P&L, per day', 'pnl_timing'),
                      ('the raw-beta line', 'book_beta, market_return, pnl_beta'),
                      ('realized against expected cost', 'realized_cost_bps, expected_cost_bps'),
                      ('forecast against realized vol', 'forecast_vol, realized_vol'),
                      ('fill differences', 'n_target, n_filled, max_fill_gap')):
    print(f'  {panel:30s} {column}')

  the schema declares attribution as required: True
  the builder emits exactly its keys: True
  cumulative sums are the artifact's own sums: True

  panel                          source column
  cumulative total/factor/idio/cost pnl_total, pnl_factor, pnl_idio, pnl_cost
  worst day's identity residual  identity_residual
  the hedge's factor P&L, per day pnl_timing
  the raw-beta line              book_beta, market_return, pnl_beta
  realized against expected cost realized_cost_bps, expected_cost_bps
  forecast against realized vol  forecast_vol, realized_vol
  fill differences               n_target, n_filled, max_fill_gap


## 9. What would falsify this, and what is not here

Falsification is written before the live numbers, so the report cannot be read
as a story told afterwards. The memo carries the full list; the three that
matter are that the identity fails on live days, that the two estimators
disagree beyond one standard error on a book that rebalances daily, and that
the hedge's own factor P&L grows rather than shrinks.

Named as open rather than quietly dropped: the regime table and the seven-way
selection, sizing and timing decomposition are in the roadmap's E12 scope and
are not among this sprint's nine tasks.

In [11]:
# Closing checklist: the criteria appear, and what is pending is pending.
joined = json.dumps(results)
for name in ('F12.1', 'F12.2', 'F12.3'):
    assert name in joined, name
assert results['criteria']['F12.3']['expected_verdict'] == 'luck'
print('  F12.1', results['criteria']['F12.1']['verdict'])
print('  F12.2', results['criteria']['F12.2']['verdict'])
print('  F12.3', results['criteria']['F12.3']['verdict'],
      '| expected verdict:', results['criteria']['F12.3']['expected_verdict'])
print('closing checklist: clean')

  F12.1 partial
  F12.2 pending
  F12.3 pending | expected verdict: luck
closing checklist: clean
